
## 03_gold_aggregate

Gold layer for GitHub event data.
Reads from Silver Delta → produces 3 aggregation tables:

  1. gold_github_summary     → daily event type counts
  2. gold_repo_activity      → top repos by activity
  3. gold_actor_activity     → top contributors by activity

Why 3 tables?
  Different consumers need different grains:
  - Executives want daily totals (table 1)
  - Repo owners want repo-level stats (table 2)
  - HR/community wants contributor stats (table 3)

Study Note:
  In production these would be separate Gold tables
  served to different teams via OneLake Shortcuts
  or Unity Catalog access controls.

In [0]:
%python

# Define config variables inline
SILVER_TABLE = "default.silver_github_events"

from pyspark.sql import functions as F
from delta.tables import DeltaTable
from datetime import datetime

# Gold table names
GOLD_SUMMARY_TABLE = "default.gold_github_summary"
GOLD_REPO_TABLE    = "default.gold_repo_activity"
GOLD_ACTOR_TABLE   = "default.gold_actor_activity"

print("Libraries loaded ✓")
print(f"Source : {SILVER_TABLE}")
print(f"Gold 1 : {GOLD_SUMMARY_TABLE}")
print(f"Gold 2 : {GOLD_REPO_TABLE}")
print(f"Gold 3 : {GOLD_ACTOR_TABLE}")

In [0]:
%python

# ── Read Silver GitHub events ─────────────────────────────────
# STUDY NOTE:
#   Same pattern as Project 1 — read by TABLE NAME not path.
#   Silver has 725,715 events across 5 hourly files.
#   Gold doesn't care how many files Silver has —
#   it just reads the registered Delta table.
#   This is the benefit of the catalog registration pattern.

df_silver = spark.table(SILVER_TABLE)

print(f"Silver rows    : {df_silver.count():,}")
print(f"Silver columns : {df_silver.columns}")
print(f"Date range     :")
df_silver.select(
    F.min("event_date").alias("min_date"),
    F.max("event_date").alias("max_date")
).show()

In [0]:
%python

# ── Gold 1: Daily event type summary ─────────────────────────
# STUDY NOTE:
#   Grain = one row per event_date × event_type
#   This answers: "How many PushEvents happened on Jan 1?"
#   Classic time-series aggregation for dashboards.
#
#   unique_actors = COUNT DISTINCT of actor_login
#   This is a HyperLogLog approximation in Spark —
#   approx_count_distinct is faster than COUNT DISTINCT
#   on large datasets with ~2% error margin.
#   PRO: Much faster on large datasets
#   CON: Approximate, not exact
#   For exact counts use F.countDistinct() — slower

df_gold_summary = df_silver.groupBy(
    "event_date",
    "event_type"
).agg(
    F.count("*")                            .alias("total_events"),
    F.approx_count_distinct("actor_login")  .alias("unique_actors"),
    F.approx_count_distinct("repo_name")    .alias("unique_repos"),
    F.current_timestamp()                   .alias("last_updated")
)

print(f"Gold Summary rows : {df_gold_summary.count():,}")
print("Sample:")
display(
    df_gold_summary
    .orderBy(F.desc("total_events"))
    .limit(10)
)

In [0]:
%python

# ── Gold 2: Top repos by activity ────────────────────────────
# STUDY NOTE:
#   Grain = one row per repo_name × event_date
#   This answers: "Which repos were most active today?"
#
#   We use collect_set to gather all event types
#   that happened on a repo in a day.
#   collect_set returns a unique array of values.
#   Example: ["PushEvent", "PullRequestEvent", "WatchEvent"]
#
#   This is an ARRAY column — a non-scalar type.
#   PRO: Rich information in one column
#   CON: Harder to query with SQL, not Power BI friendly
#   In production you'd explode this or keep as metadata only

df_gold_repo = df_silver \
    .filter(F.col("repo_name").isNotNull()) \
    .groupBy("repo_name", "event_date") \
    .agg(
        F.count("*")                            .alias("total_events"),
        F.approx_count_distinct("actor_login")  .alias("unique_contributors"),
        F.collect_set("event_type")             .alias("event_types"),
        F.current_timestamp()                   .alias("last_updated")
    )

print(f"Gold Repo rows : {df_gold_repo.count():,}")
print("\nTop 10 most active repos:")
display(
    df_gold_repo
    .orderBy(F.desc("total_events"))
    .limit(10)
)

In [0]:
%python

# ── Gold 3: Top actors (contributors) by activity ─────────────
# STUDY NOTE:
#   Grain = one row per actor_login × event_date
#   This answers: "Who were the most active GitHub users today?"
#
#   push_count filters for PushEvent only —
#   this is a conditional aggregation pattern:
#   F.when(condition, value_if_true).otherwise(value_if_false)
#   Wrapped in F.sum() → counts only matching rows
#   This avoids needing a separate filter + join.
#   Very common pattern in Gold layer aggregations.

df_gold_actor = df_silver \
    .filter(F.col("actor_login").isNotNull()) \
    .groupBy("actor_login", "event_date") \
    .agg(
        F.count("*")                        .alias("total_events"),
        F.approx_count_distinct("repo_name").alias("repos_touched"),
        F.sum(
            F.when(F.col("event_type") == "PushEvent", 1)
             .otherwise(0)
        )                                   .alias("push_count"),
        F.sum(
            F.when(F.col("event_type") == "PullRequestEvent", 1)
             .otherwise(0)
        )                                   .alias("pr_count"),
        F.sum(
            F.when(F.col("event_type") == "WatchEvent", 1)
             .otherwise(0)
        )                                   .alias("star_count"),
        F.current_timestamp()               .alias("last_updated")
    )

print(f"Gold Actor rows : {df_gold_actor.count():,}")
print("\nTop 10 most active contributors:")
display(
    df_gold_actor
    .orderBy(F.desc("total_events"))
    .limit(10)
)

In [0]:
%python

# ── Write Gold tables using MERGE ────────────────────────────
# STUDY NOTE:
#   Same MERGE pattern as Project 1 but with different keys:
#   Summary → merge on event_date + event_type
#   Repo    → merge on repo_name + event_date
#   Actor   → merge on actor_login + event_date
#
#   Why MERGE here too?
#   Each time Auto Loader processes a new Bronze file,
#   we re-aggregate Silver and update Gold.
#   MERGE ensures new hours UPDATE existing day totals
#   rather than inserting duplicate date rows.

def write_gold_merge(df, table_name, merge_keys: list):
    """
    Generic MERGE writer for Gold tables.
    merge_keys: list of column names to match on
    """
    if not spark.catalog.tableExists(table_name):
        print(f"Creating: {table_name}")
        df.write.format("delta").mode("overwrite") \
          .saveAsTable(table_name)
        print(f"✓ Created: {table_name} ({df.count():,} rows)")
    else:
        print(f"Merging : {table_name}")
        delta_tbl = DeltaTable.forName(spark, table_name)

        # Build merge condition from keys
        merge_condition = " AND ".join([
            f"target.{k} = source.{k}" for k in merge_keys
        ])

        delta_tbl.alias("target").merge(
            df.alias("source"),
            merge_condition
        ).whenMatchedUpdateAll() \
         .whenNotMatchedInsertAll() \
         .execute()

        count = spark.table(table_name).count()
        print(f"✓ Merged: {table_name} ({count:,} rows)")

print(f"Writing Gold tables...")
print(f"Started: {datetime.now()}\n")

write_gold_merge(
    df_gold_summary,
    GOLD_SUMMARY_TABLE,
    ["event_date", "event_type"]
)

write_gold_merge(
    df_gold_repo,
    GOLD_REPO_TABLE,
    ["repo_name", "event_date"]
)

write_gold_merge(
    df_gold_actor,
    GOLD_ACTOR_TABLE,
    ["actor_login", "event_date"]
)

print(f"\nCompleted: {datetime.now()}")

In [0]:
%python

# ── Final verification ────────────────────────────────────────
print("=" * 50)
print("Gold Layer Summary")
print("=" * 50)

for table in [GOLD_SUMMARY_TABLE, GOLD_REPO_TABLE, GOLD_ACTOR_TABLE]:
    count = spark.table(table).count()
    cols  = len(spark.table(table).columns)
    print(f"{table:<40} {count:>8,} rows  {cols} cols")

print("=" * 50)

# Show event type distribution
print("\nEvent distribution in Gold Summary:")
display(
    spark.table(GOLD_SUMMARY_TABLE)
    .orderBy(F.desc("total_events"))
)